# Part 0. Import libraries and data

In [19]:
import pandas as pd
import numpy as np
from numpy import linalg as la
import math
from collections import defaultdict, Counter
from array import array
import ast
from nltk.stem import PorterStemmer
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\zsy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [20]:
filename = "../../data/index_info.csv"
index_info = pd.read_csv(filename)

# List of fields, excluding 'pid': 'title', 'description', etc.
fields = index_info.columns.to_list()
fields.remove('pid')

# Parse fields as lists of terms instead of plain strings
for f in fields:
    index_info[f] = index_info[f].apply(ast.literal_eval)

# Sort by PID
index_info = index_info.sort_values('pid').reset_index(drop=True)

index_info.head(3)

,pid,title,description,brand,category,sub_category,seller,product_details
0,4hWtdShnw%3D%3D,"[men, regular, fit, solid, casual, shirt]",[],"[crock, cl]","[cloth, accessori]",[topwear],[crockscl],"[1, c328, 2, 38, regular, pure, cotton, full, ..."
1,ARME8P6GFSXGE9AW,"[welwear, suncool1008, nylon, arm, warmer, mul...","[tatto, sleev, protect, uv, raditaion, also, p...",[],"[cloth, accessori]","[winter, wear]","[navkar, craft]","[2, 2, arm, warmer, welwear, suncool1008, brow..."
2,ARME8P6GJQ8PZBE6,"[welwear, suncool1004, nylon, arm, warmer, mul...","[tatto, sleev, protect, uv, raditaion, also, p...",[],"[cloth, accessori]","[winter, wear]","[navkar, craft]","[1, 1, arm, warmer, welwear, suncool1004, blue..."


# Part 1: Indexing

## 1. Index creation

In [21]:
def create_index(index_df: pd.DataFrame) -> defaultdict:  
    """
    Implement the inverted index
    
    :param index_df: DataFrame storing pre-processed product information 
    :returns index: the inverted index (implemented through a nested Python dictionary) containing terms and fields as keys and the corresponding
    list of documents where these keys appears in (and the positions) as values.
    """
    index = defaultdict(dict)

    # Iterate over each row (products)
    for i, row in index_df.iterrows():
        pid = row['pid'] # Product's ID

        # Build index only for this product and field
        # E.g. ['solid', 'women'] --> 'solid': {'title': [pid, [0]]}, 'women': {'title': [pid, [1]]}
        current_product_index = {}

        # Iterate over fields in this row: title, description, brand, ...
        for field in fields:

            terms = row[field] # List of preprocessed terms

            for position, term in enumerate(terms): # terms contains page_title + page_text. Loop over all terms
                if term in current_product_index.keys():

                    if field in current_product_index[term].keys():
                        # if the term is already in the current_product_index for the current page (current_page_current_product_index)
                        # append the position to the corresponding list
                        current_product_index[term][field][1].append(position)
                    else:
                        current_product_index[term][field] = [pid, array('I',[position])]              
                
                else:
                    # term is not in the dictionary yet
                    current_product_index[term] = {field: [pid, array('I',[position])]} #'I' indicates unsigned int (int in Python)}
                        
        # Merge the current product index with the main index
        for term, field_dict in current_product_index.items():
            for field, posting_page in field_dict.items():
                index[term].setdefault(field, []).append(posting_page)

    return index

index = create_index(index_info)

# for term, postings in list(index.items())[:1]:
#     print(term, postings)

## 2. Test queries

Our extremely creative (😉) test queries will be:
1. `men's cotton checkered shirt`
2. `multicolor women casual t-shirt`
3. `black beat sports shoes`
4. `animal print jacket for women`
5. `grey hoodie with pockets`

In [22]:
# Preprocess the text of the description column
def build_terms(text):
    """
    Preprocess text removing stop words, stemming,
    transforming in lowercase and return the tokens of the text.

    :param text: to be preprocessed
    :returns: a list of tokens corresponding to the input text after the preprocessing
    """

    stemmer = PorterStemmer()
    tokenizer = RegexpTokenizer(r'\w+')             # keep only words & numbers, drop punctuation
    stop_words = set(stopwords.words("english"))
    for w in ['not', 'no']:
        stop_words.remove(w)                        # keep 'no' and 'not' to preserve meaning of sentences like "shirt with no sleeves"
    text= text.lower()                              # transform in lowercase
    text=  tokenizer.tokenize(text)                 # tokenize the text to get a list of terms
    text=[t for t in text if t not in stop_words]   # eliminate the stopwords
    text=[stemmer.stem(t) for t in text]            # perform stemming
    return text

In [23]:
def search(query, index):
    """
    The output is the list of documents that contain ALL of the query terms.
    So, we will get the list of documents for each query term, and take the intersection of them.
    """
    query = build_terms(query)

    ################ Add Empty String Case  ################
    if not query:
        return []
    
    for i, term in enumerate(query):

        # term is not in index, there is no product with this query term
        if term not in index.keys():
            return []

        # store in term_pids the ids of the products that contain "term" in any field
        term_pids = set()
        for field in index[term].keys():
            for posting in index[term][field]:
                pid = posting[0]
                term_pids.add(pid)
        
        # docs = intersection of term_pids
        if i == 0:
            docs = term_pids
        else:
            docs = docs.intersection(term_pids)
            
    docs = list(docs)
    return docs

In [24]:
matches_cotton = search("cotton", index)
matches_pants = search("pants", index)
matches_both = search("cotton and pants", index)

set(matches_cotton) & set(matches_pants) == set(matches_both)

True

In [41]:
# TODO: search for test queries using "search" function
test_queries = [
    "men's cotton checkered shirt",
    "multicolor women casual t-shirt",
    "black beat sports shoes",
    "animal print jacket for women",
    "grey hoodie with pockets",
]

for query in test_queries:
    docs = search(query, index)
    ratio = len(docs) / len(index_info)
    
    print(
        f"{query!r} : {len(docs)} products, "
        f"match rate = {ratio:.2%}"
    )

"men's cotton checkered shirt" : 401 products, match rate = 1.43%
'multicolor women casual t-shirt' : 680 products, match rate = 2.42%
'black beat sports shoes' : 0 products, match rate = 0.00%
'animal print jacket for women' : 1 products, match rate = 0.00%
'grey hoodie with pockets' : 28 products, match rate = 0.10%


## 3. TF-IDF algorithm

Our index structure:

```
index["cotton"] = {
    "title": [
        [docA, positions],
        [docB, positions],
    ],
    "description": [
        [docA, positions],
    ],
    ...
}
```

In [ ]:
def create_index_tfidf(index_df: pd.DataFrame, num_documents: int):  
    """
    Implement the inverted index and compute tf, df and idf
    
    :param index_df: DataFrame storing pre-processed product information 
    :returns index: the inverted index (implemented through a nested Python dictionary) containing terms and fields as keys and the corresponding
    list of documents where these keys appears in (and the positions) as values.
    :returns tf: normalized term frequency for each term in each document
    :returns df: number of documents each term appear in
    :returns idf: inverse document frequency of each term
    """
    index = defaultdict(dict)
    ######## We need to create a dictionary of dictionary to store different fields #########
    ######## e.g. tf["cotton"]["title"][pid] = TF value ##########################
    tf = defaultdict(lambda: defaultdict(dict)) 
    # tf = defaultdict(list)  # term frequencies of terms in documents (documents in the same order as in the main index)
    df = defaultdict(int)  # document frequencies of terms in the corpus
    idf = defaultdict(float)

    # Iterate over each row (products)
    for i, row in index_df.iterrows():
        pid = row['pid'] # Product's ID

        # Build index only for this product and field
        # E.g. ['solid', 'women'] --> 'solid': {'title': [pid, [0]]}, 'women': {'title': [pid, [1]]}
        current_product_index = {}

        # Iterate over fields in this row: title, description, brand, ...
        for field in fields:

            terms = row[field] # List of preprocessed terms

            for position, term in enumerate(terms): # terms contains page_title + page_text. Loop over all terms
                if term in current_product_index.keys():

                    if field in current_product_index[term].keys():
                        # if the term is already in the current_product_index for the current page (current_page_current_product_index)
                        # append the position to the corresponding list
                        current_product_index[term][field][1].append(position)
                    else:
                        current_product_index[term][field] = [pid, array('I',[position])]              
                
                else:
                    # term is not in the dictionary yet
                    current_product_index[term] = {field: [pid, array('I',[position])]} #'I' indicates unsigned int (int in Python)}
                        
        # Normalize term frequencies
        # Compute the denominator to normalize term frequencies
        # norm is the same for all terms of a document.
        ##################### Doc length normalization formula is sqrt(sum of (term frequency)^2) ############
        ##################### This is summinig over each term's square individually #######################################
        ##################### I have changed the indexing methods to compute ##########################
        norm = 0
        # for term in current_product_index.keys():
        #     for field, posting in current_product_index[term].items():
        #         # posting will contain the list of positions for current term in current document.
        #         # posting ==> [current_doc, [list of positions]]
        #         # you can use it to infer the frequency of current term.
        #         norm += len(posting[1]) ** 2
        # norm = math.sqrt(norm)
        for term in current_product_index.keys():
            # sum over all the fields for the current term to get the total frequency of the term in the current document
            norm += sum(
                len(posting[1])
                for posting in current_product_index[term].values()
            ) ** 2
        norm = math.sqrt(norm)

        # calculate the tf(dividing the term frequency by the above computed norm) and df weights
        # for term in current_product_index.keys():
        #     for field, posting in current_product_index[term].items():
        #         # append the tf for current term (tf = term frequency in current doc/norm)
        #         tf[term].append(np.round(len(posting[1])/norm, 4)) ## SEE formula (1) above
        #         #increment the document frequency of current term (number of documents containing the current term)
                
        #     df[term] += 1 # increment DF for current term
        
        # 
        for term in current_product_index.keys():
            for field, posting in current_product_index[term].items():
                # append the tf for current term (tf = term frequency in current doc/norm)
                tf[term][field][pid] = np.round(len(posting[1])/norm, 4) 
        
            df[term] += 1 # increment DF for current term
            
        # Merge the current product index with the main index
        for term, field_dict in current_product_index.items():
            for field, posting_page in field_dict.items():
                index[term].setdefault(field, []).append(posting_page)

    # Compute IDF following the formula (3) above. HINT: use np.log
    # Note: It is computed later after we know the df.
    for term in df:
        idf[term] = np.round(np.log(float(num_documents/df[term])), 4)

    return index, tf, df, idf

index, tf, df, idf = create_index_tfidf(index_info, num_documents=len(index_info))

In [27]:
# For now, we give equal weights to all fields (1/7)
field_weights = np.array(len(fields)*[1/len(fields)])
field_weights

array([0.14285714, 0.14285714, 0.14285714, 0.14285714, 0.14285714,
       0.14285714, 0.14285714])

In [28]:
# TODO: finish this part
# implement field weights (if all weights are equal the result should be the same as regular TF-IDF)

# TODO: create title_index, a dictionary where keys are product PIDS and values are the original titles

def rank_documents(terms, docs, index, idf, tf):
    """
    Perform the ranking of the results of a search based on the tf-idf weights

    Argument:
    terms -- list of query terms
    docs -- list of documents, to rank, matching the query
    index -- inverted index data structure
    idf -- inverted document frequencies
    tf -- term frequencies

    Returns:
    Print the list of ranked documents
    """

    
    ################## Add Empty query or no doc case ################
    if not terms or not docs:
        return []
    
    # I'm interested only on the element of the docVector corresponding to the query terms
    # The remaining elements would became 0 when multiplied to the query_vector
    
    doc_vectors = defaultdict(lambda: [0] * len(terms)) # I call doc_vectors[k] for a nonexistent key k, the key-value pair (k,[0]*len(terms)) will be automatically added to the dictionary
    query_vector = [0] * len(terms)

    # compute the norm for the query tf
    query_terms_count = Counter(terms)  # get the frequency of each term in the query.
    # Example: collections.Counter(["hello","hello","world"]) --> Counter({'hello': 2, 'world': 1})

    query_norm = la.norm(list(query_terms_count.values()))

    ############## I have changed terms to query_terms_count to avoid duplicates in the query vector##################
    for termIndex, term in enumerate(query_terms_count):  #termIndex is the index of the term in the query
        if term not in index:
            continue

        ## Compute tf*idf(normalize TF as done with documents)
        query_vector[termIndex]=query_terms_count[term]/query_norm * idf[term] #query_vector[0] corresponds to the first term in the query

        # Generate doc_vectors for matching docs
        # index[term][field] contains [doc, positions] records.
        for field, postings in index[term].items():
            for doc, positions in postings:
                if doc in docs:
                    # Sum TF-IDF contributions from all fields equally.
                    doc_vectors[doc][termIndex] += (
                        tf[term][field][doc] * idf[term]
                    )
        
        # for doc_index, (doc, postings) in enumerate(index[term]):

        #     #tf[term][0] will contain the tf of the term "term" in the doc 26
        #     if doc in docs: #if the odcument is in the list of documents retrieved (matching the query)
        #         doc_vectors[doc][termIndex] = tf[term][doc_index] * idf[term]  # TODO: check if multiply for idf
        

    # Calculate the score of each doc

    # compute the cosine similarity between queyVector and each docVector:
    # HINT: you can use the dot product because in case of normalized vectors it corresponds to the cosine similarity
    # see np.dot
    
    doc_scores=[[np.dot(curDocVec, query_vector), doc] for doc, curDocVec in doc_vectors.items() ]
    doc_scores.sort(reverse=True)
    result_docs = [x[1] for x in doc_scores]
    return result_docs

In [ ]:
def search_tf_idf(query, index):
    """
    output is the list of documents that contain any of the query terms.
    So, we will get the list of documents for each query term, and take the union of them.
    """
    docs = search(query, index)
    query = build_terms(query)

    ranked_docs = rank_documents(query, docs, index, idf, tf)
    return ranked_docs

In [38]:
# test queries
products = pd.read_csv("../../data/fashion_products.csv")
title_index = products.set_index("pid")["title"].to_dict()
print("Insert your query (i.e.: women tshirt):\n")
query = input()
ranked_docs = search_tf_idf(query, index)
top = 10

print("\n======================\nTop {} results out of {} for the searched query:\n".format(top, len(ranked_docs)))
for d_id in ranked_docs[:top]:
    print("page_id= {} - page_title: {}".format(d_id, title_index[d_id]))

Insert your query (i.e.: women tshirt):


Top 10 results out of 45 for the searched query:

page_id= TSHFNV35W6XMETKT - page_title: Printed Men Round Neck Blue T-Shirt
page_id= TKPFPBCZARJAYW5G - page_title: Solid Men Blue Track Pants
page_id= TKPFNV3EBFWFBWCV - page_title: Solid Men Blue Track Pants
page_id= TKPFZGQVKAQZZWAQ - page_title: Solid Men Dark Blue Track Pants
page_id= TKPFNV3GVUHG5U7E - page_title: Solid Men Blue Track Pants
page_id= TKPFNV3EVZGQYBNZ - page_title: Solid Men Blue Track Pants
page_id= TKPFNV3ESGN9VM8D - page_title: Solid Men Blue Track Pants
page_id= TKSFZYP8WUAPARF8 - page_title: Solid Men Track Suit
page_id= TKSFZ9SP4HZJEYAA - page_title: Solid Men Track Suit
page_id= TKSFZ9R4QRSZZNGG - page_title: Self Design Men Track Suit


# Part 2

In [58]:
#TODO: Review these functions copied from Lab2 

# i. Precision@K (P@K)
def precision_at_k(doc_score, y_score, k=10):
    """
    Parameters
    ----------
    doc_score: Ground truth (true relevance labels).
    y_score: Predicted scores.
    k : number of doc to consider.

    Returns
    -------
    precision @k : float

    """
    order = np.argsort(y_score)[::-1]  # Bonus point: How to improve the effitiency of this part?
    #doc_score = np.take(doc_score, order[:k])
    doc_score = doc_score[order[:k]]
    relevant = sum(doc_score == 1)
    return float(relevant) / k

In [59]:
# ii. Recall@K (R@K)
def recall_at_k(doc_score, y_score, k=10, number_of_relevant=None):
    """Use the total known relevant count when doc_score contains only retrieved products."""
    doc_score = np.asarray(doc_score)
    if number_of_relevant is None:
        number_of_relevant = np.sum(doc_score == 1)
    if number_of_relevant == 0:
        return 0.0

    order = np.argsort(y_score)[::-1]
    doc_score = doc_score[order[:k]]
    relevant = np.sum(doc_score == 1)
    return float(relevant) / number_of_relevant


In [60]:
# iii. Average Precision@K (P@K)
def avg_precision_at_k(doc_score, y_score, k=10):
    """
    Parameters
    ----------
    doc_score: Ground truth (true relevance labels).
    y_score: Predicted scores.
    k : number of doc to consider.

    Returns
    -------
    average precision @k : float
    """
    order = np.argsort(y_score)[::-1]  # get the list of indexes of the predicted score sorted in descending order.

    prec_at_i = 0
    prec_at_i_list = []
    number_of_relevant = 0
    number_to_iterate = min(k, len(order))

    for i in range(number_to_iterate):
        if doc_score[order[i]] == 1:
            number_of_relevant += 1
            prec_at_i = number_of_relevant / (i + 1)
            prec_at_i_list.append(prec_at_i)

    if number_of_relevant == 0:
        return 0
    else:
      return np.sum(prec_at_i_list) / number_of_relevant

In [62]:
# iv. F1-Score@K
def f1_at_k(doc_score, y_score, k=10, number_of_relevant=None):
    """Harmonic mean of precision@k and recall@k."""
    doc_score = np.asarray(doc_score)
    precision = precision_at_k(doc_score, y_score, k)
    recall = recall_at_k(doc_score, y_score, k, number_of_relevant)

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


In [64]:
# v. Mean Average Precision (MAP)
def map_at_k(search_res, k=10):
    """
    Parameters
    ----------
    search_res: search results dataset containing:
        query_id: query id.
        doc_id: document id.
        predicted_relevance: relevance predicted through LightGBM.
        doc_score: actual score of the document for the query (ground truth).

    Returns
    -------
    mean average precision @ k : float
    """
    avp = []
    for q in search_res["query_id"].unique():  # loop over all query id
        curr_data = search_res[search_res["query_id"] == q]  # select data for current query
        avp.append(avg_precision_at_k(np.array(curr_data["is_relevant"]),
                   np.array(curr_data["predicted_relevance"]), k))  #append average precision for current query
    return np.sum(avp) / len(avp), avp  # return mean average precision

In [65]:
# vi. Mean Reciprocal Rank (MRR)
def rr_at_k(doc_score, y_score, k=10):
    """
    Parameters
    ----------
    doc_score: Ground truth (true relevance labels).
    y_score: Predicted scores.
    k : number of doc to consider.

    Returns
    -------
    Reciprocal Rank for qurrent query
    """

    order = np.argsort(y_score)[::-1]  # get the list of indexes of the predicted score sorted in descending order.
    doc_score = np.take(doc_score, order[:k])  # sort the actual relevance label of the documents based on predicted score(hint: np.take) and take first k.
    if np.sum(doc_score) == 0:  # if there are not relevant doument return 0
        return 0
    return 1.0 / (np.argmax(doc_score == 1) + 1)  # hint: to get the position of the first relevant document use "np.argmax"


def mrr_at_k(search_res, k=10):
    """Mean RR@k across queries; columns match map_at_k."""
    rr = []
    for q in search_res["query_id"].unique():
        curr_data = search_res[search_res["query_id"] == q]
        rr.append(rr_at_k(
            np.array(curr_data["is_relevant"]),
            np.array(curr_data["predicted_relevance"]), k
        ))

    if len(rr) == 0:
        return 0.0, rr
    return np.mean(rr), rr


In [66]:
# vii. Normalized Discounted Cumulative Gain (NDCG)
def dcg_at_k(doc_score, y_score, k=10):
    order = np.argsort(y_score)[::-1]  # get the list of indexes of the predicted score sorted in descending order.
    doc_score = np.take(doc_score, order[:k])  # sort the actual relevance label of the documents based on predicted score(hint: np.take) and take first k.
    gain = 2 ** doc_score - 1  # Compute gain (use formula 7 above)
    discounts = np.log2(np.arange(len(doc_score)) + 2)  # Compute denominator
    return np.sum(gain / discounts)  #return dcg@k


def ndcg_at_k(doc_score, y_score, k=10):
    dcg_max = dcg_at_k(doc_score, doc_score, k)
    if not dcg_max:
        return 0
    return np.round(dcg_at_k(doc_score, y_score, k) / dcg_max, 4)

In [ ]:

#TODO REVIEW THIS PART
########### IM NOT SURE IF THE CORPUS IS THE ENTIRE DATASET OR######################
############  JUST THE VALIDATION SET. NOW IT'S IMPLEMENTED IN THE ENTIRE DATASET. ############
############ THE SCORE OF USING WHOLE DATASET IS TOO LOW ###########################


validation_labels = pd.read_csv("../../data/validation_labels.csv")

queries = {
    1: "women full sleeve sweatshirt cotton",
    2: "men slim jeans blue",
}

# Evaluate the top 10 results for each query. which will be treated as relevant by the algorithm
k = 10
# Collect per-query data for MAP@K and MRR@K
search_results = []

for query_id, query in queries.items():
    # Select the labeled products for the current query
    curr_data = validation_labels[
        validation_labels["query_id"] == query_id
    ]
    
    # Map each product PID to its ground-truth label: 1 = relevant, 0 = nonrelevant
    labels = curr_data.set_index("pid")["labels"].to_dict()

    # Count all known relevant products, including those not retrieved.
    number_of_relevant = sum(label == 1 for label in labels.values())

    # Search the full corpus and rank all matching products
    ranked_docs = search_tf_idf(query, index)

    
    # Evaluation convention: treat unjudged products as nonrelevant
    doc_score = np.array(
        [labels.get(doc, 0) for doc in ranked_docs],
        dtype=int
    )
    
    # Preserve the returned ranking
    y_score = np.arange(len(ranked_docs), 0, -1, dtype=float)

    values = [
        precision_at_k(doc_score, y_score, k),
        recall_at_k(doc_score, y_score, k, number_of_relevant),
        avg_precision_at_k(doc_score, y_score, k),
        f1_at_k(doc_score, y_score, k, number_of_relevant),
        rr_at_k(doc_score, y_score, k),
        ndcg_at_k(doc_score, y_score, k),
    ]

    print(" ".join(f"{value:.3f}" for value in values))

    search_results.append(pd.DataFrame({
        "query_id": query_id,
        "is_relevant": doc_score,
        "predicted_relevance": y_score,
    }))

search_results = pd.concat(search_results, ignore_index=True)

map_score, _ = map_at_k(search_results, k)
mrr_score, _ = mrr_at_k(search_results, k)

print(f"{map_score:.3f} {mrr_score:.3f}")

0.100 0.077 0.200 0.087 0.200 0.085
0.100 0.100 0.333 0.100 0.333 0.151
0.267 0.267
